# Clase 3: generalización, selección de modelos y regularización

Visualizaciones interactivas para acompañar las notas de la clase 3. Las figuras siguen el mismo orden y los mismos colores de las notas, y usan exactamente los mismos datos: se importan desde `notes/scripts/03-regularizacion-figuras.py`, el script que genera las figuras del PDF.

Para usarlo en clase basta con ejecutar todas las celdas al inicio y luego mover los sliders.

In [ ]:
%matplotlib inline
import importlib.util
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact, widgets
from matplotlib.colors import ListedColormap

# Datos y ajustes de las notas (los mismos que generan las figuras del PDF)
ruta = Path("../notes/scripts/03-regularizacion-figuras.py")
spec = importlib.util.spec_from_file_location("fig03", ruta)
fig03 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fig03)

AZUL, VERDE, ROJO, NARANJA, GRIS = "tab:blue", "tab:green", "tab:red", "tab:orange", "0.55"
CELESTE, DURAZNO = "#bcd4ee", "#f6c28b"

plt.rcParams.update({
    "figure.figsize": (11, 4.2),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

## 1. Regresión polinomial (Figura 1)

Ajustamos polinomios de grado $d$ a los mismos 15 ejemplos de entrenamiento de las notas, generados a partir de $f(x) = 2 + 0.5x + \sin(1.3x)$ con ruido.

Qué observar al mover el grado:

- El error de entrenamiento (gris punteado) baja siempre: un polinomio de grado $d+1$ puede imitar a cualquiera de grado $d$.
- El error de validación (rojo) baja hasta grado 3 o 4 y luego sube. Con grado 9 o 10 es enorme.
- El título muestra el coeficiente más grande en valor absoluto. Con grados altos los coeficientes explotan, y esa es la idea que retoma la regularización.

In [ ]:
curva_poli = fig03.curva_error_polinomios()
grados = [r["grado"] for r in curva_poli]
mse_train_poli = [r["mse_train"] for r in curva_poli]
mse_val_poli = [r["mse_val"] for r in curva_poli]


def estilo_grado(grado):
    if grado <= 2:
        return dict(color=AZUL, ls="--")
    if grado <= 6:
        return dict(color=VERDE, ls="-")
    return dict(color=ROJO, ls="-")


def ver_polinomio(grado=1):
    x = fig03.x_grid
    theta = fig03.ajustar_ols(grado)
    fig, (ax1, ax2) = plt.subplots(1, 2)

    ax1.plot(x, fig03.f_real(x), color=GRIS, lw=5, alpha=0.5, label="función real")
    ax1.plot(x, fig03.potencias(x, grado) @ theta, lw=2, label=f"grado {grado}", **estilo_grado(grado))
    ax1.scatter(fig03.x_train, fig03.y_train, color="black", zorder=3, label="entrenamiento")
    ax1.set(xlim=(0, 5), ylim=(0, 5), xlabel="$x$", ylabel="$y$",
            title=f"Grado {grado}:  max|θ| = {np.abs(theta).max():.3g}")
    ax1.legend(loc="upper left", fontsize=9)

    ax2.semilogy(grados, mse_train_poli, "--", color=GRIS, lw=2, label="entrenamiento")
    ax2.semilogy(grados, mse_val_poli, color=ROJO, lw=2, label="validación")
    ax2.axhline(fig03.SIGMA**2, color="0.7", ls=":", label="varianza del ruido")
    ax2.plot(grado, mse_train_poli[grado - 1], "o", color=GRIS, ms=8)
    ax2.plot(grado, mse_val_poli[grado - 1], "o", color=ROJO, ms=8)
    ax2.set(xlabel="Grado del polinomio", ylabel="MSE", xticks=grados)
    ax2.legend(fontsize=9)
    plt.show()

In [ ]:
interact(ver_polinomio, grado=widgets.IntSlider(min=1, max=10, value=1, description="Grado"));

## 2. Validación cruzada (Figura 3)

Ahora escogemos el grado sin usar el conjunto de validación grande: solo con los 15 ejemplos de entrenamiento y validación cruzada k-fold. Los ejemplos se dividen en $k$ partes; en cada iteración una parte (naranja) se usa para validar y las demás (azul) para entrenar. El error estimado es el promedio de los $k$ errores. Con $k = 15$ cada parte tiene un solo ejemplo: es leave-one-out (LOOCV).

Qué observar:

- A la izquierda, cómo cambia la partición con $k$.
- A la derecha, el error de validación cruzada para cada grado (rojo), comparado con el error medido sobre 300 ejemplos nuevos (gris). La estrella marca el grado que escogería la validación cruzada.
- Aunque solo usa 15 ejemplos, la validación cruzada tiene la misma forma de U y escoge un grado razonable.

In [ ]:
m_poli = len(fig03.x_train)
orden = np.random.default_rng(0).permutation(m_poli)  # asignación de ejemplos a las partes


def error_cv(k):
    partes = np.array_split(orden, k)
    errores = []
    for g in grados:
        e = []
        for val in partes:
            ent = np.setdiff1d(orden, val)
            X = fig03.potencias(fig03.x_train[ent], g)
            theta, *_ = np.linalg.lstsq(X, fig03.y_train[ent], rcond=None)
            e.append(fig03.mse(fig03.potencias(fig03.x_train[val], g), fig03.y_train[val], theta))
        errores.append(np.mean(e))
    return np.array(errores), partes


def ver_validacion_cruzada(k=5):
    errores, partes = error_cv(k)
    fig, (ax1, ax2) = plt.subplots(1, 2, gridspec_kw=dict(width_ratios=[1.2, 1]))

    # Columnas ordenadas por parte, para que cada parte se vea como un bloque
    tabla = np.zeros((k, m_poli))
    inicio = 0
    for i, p in enumerate(partes):
        tabla[i, inicio:inicio + len(p)] = 1
        inicio += len(p)
    ax1.imshow(tabla, cmap=ListedColormap([CELESTE, DURAZNO]), aspect="auto")
    ax1.set_xticks(np.arange(-0.5, m_poli, 1), minor=True)
    ax1.set_yticks(np.arange(-0.5, k, 1), minor=True)
    ax1.grid(which="minor", color="white", lw=1.5)
    ax1.tick_params(which="minor", length=0)
    ax1.set_xticks([])
    ax1.set_yticks(range(k), [f"Iteración {i + 1}" for i in range(k)], fontsize=8 if k > 8 else 10)
    ax1.set_xlabel("Ejemplos de entrenamiento")
    ax1.set_title(f"k = {k}" + ("  (LOOCV)" if k == m_poli else ""))
    for s in ax1.spines.values():
        s.set_visible(False)

    ax2.semilogy(grados, mse_val_poli, color=GRIS, lw=2, label="validación (300 ejemplos)")
    ax2.semilogy(grados, errores, "o-", color=ROJO, lw=2, label=f"validación cruzada, k = {k}")
    mejor = int(np.argmin(errores))
    ax2.plot(grados[mejor], errores[mejor], "*", color="black", ms=16, zorder=3,
             label=f"mejor grado: {grados[mejor]}")
    ax2.set(xlabel="Grado del polinomio", ylabel="MSE", xticks=grados)
    ax2.legend(fontsize=9)
    plt.show()

In [ ]:
interact(ver_validacion_cruzada, k=widgets.IntSlider(min=2, max=15, value=5, description="k"));

## 3. Ridge con una sola feature

Tenemos solo dos ejemplos de entrenamiento (negro) y algunos ejemplos de prueba (naranja). Con dos puntos, la recta de mínimos cuadrados pasa exactamente por ellos: el error de entrenamiento es cero, pero la pendiente es demasiado grande y el error de prueba es alto.

Con la feature centrada, la solución de Ridge es

$$
\theta_0 = \bar{y}, \qquad \theta_1 = \frac{\sum_i x^{(i)} y^{(i)}}{\sum_i (x^{(i)})^2 + \lambda}.
$$

Qué observar al mover $\lambda$:

- La recta gira alrededor del punto $(\bar{x}, \bar{y})$ y se va aplanando. Tiende a la recta horizontal $y = \bar{y}$, no a $y = 0$, porque el intercepto no se penaliza.
- $\lambda$ solo aparece en el denominador de $\theta_1$: mientras más grande, más pequeña la pendiente.
- A la derecha, el error de entrenamiento siempre sube con $\lambda$, mientras que el de prueba primero baja y luego sube: un poco de sesgo nos compra mucha menos varianza.

In [ ]:
def f_1d(x):
    return 1.6 + 0.4 * x


x_ent_1d = np.array([1.5, 3.5])
y_ent_1d = np.array([1.6, 3.6])
rng_1d = np.random.default_rng(3)
x_prueba_1d = np.linspace(0.6, 4.4, 8)
y_prueba_1d = f_1d(x_prueba_1d) + rng_1d.normal(0, 0.3, len(x_prueba_1d))

x_media, y_media_1d = x_ent_1d.mean(), y_ent_1d.mean()
xc = x_ent_1d - x_media  # feature centrada
numerador, suma_cuadrados = xc @ y_ent_1d, xc @ xc


def ridge_1d(lam):
    """Devuelve (θ_0, θ_1) en términos de x centrado."""
    return y_media_1d, numerador / (suma_cuadrados + lam)


def mse_1d(x, y, lam):
    th0, th1 = ridge_1d(lam)
    return np.mean((th0 + th1 * (x - x_media) - y) ** 2)


lambdas_1d = np.logspace(-2, 3, 200)
mse_ent_1d = [mse_1d(x_ent_1d, y_ent_1d, l) for l in lambdas_1d]
mse_prueba_1d = [mse_1d(x_prueba_1d, y_prueba_1d, l) for l in lambdas_1d]


def ver_ridge_1d(lam=0.01):
    th0, th1 = ridge_1d(lam)
    x = np.linspace(0, 5, 2)
    fig, (ax1, ax2) = plt.subplots(1, 2)

    ax1.plot(x, y_media_1d + numerador / suma_cuadrados * (x - x_media), color="0.8", ls=":",
             label="mínimos cuadrados (λ = 0)")
    ax1.axhline(y_media_1d, color=GRIS, lw=1, ls="--", label=r"$y = \bar{y}$")
    ax1.plot(x, th0 + th1 * (x - x_media), color=AZUL, lw=2.5, label="Ridge")
    ax1.scatter(x_prueba_1d, y_prueba_1d, color=NARANJA, s=45, zorder=3, label="prueba")
    ax1.scatter(x_ent_1d, y_ent_1d, color="black", s=60, zorder=4, label="entrenamiento")
    ax1.plot(x_media, y_media_1d, "P", color=AZUL, ms=10, mec="white", zorder=5)
    ax1.text(0.2, 4.55,
             rf"$\theta_1 = \frac{{\sum x^{{(i)}} y^{{(i)}}}}{{\sum (x^{{(i)}})^2 + \lambda}}"
             rf" = \frac{{{numerador:.2f}}}{{{suma_cuadrados:.2f} + {lam:.3g}}} = {th1:.3f}$",
             fontsize=13)
    ax1.set(xlim=(0, 5), ylim=(0, 5), xlabel="$x$", ylabel="$y$",
            title=rf"$\lambda$ = {lam:.3g}:   $\theta_0 = \bar{{y}}$ = {th0:.2f}")
    ax1.legend(loc="lower right", fontsize=9)

    ax2.semilogx(lambdas_1d, mse_ent_1d, "--", color=GRIS, lw=2, label="entrenamiento")
    ax2.semilogx(lambdas_1d, mse_prueba_1d, color=ROJO, lw=2, label="prueba")
    ax2.plot(lam, mse_1d(x_ent_1d, y_ent_1d, lam), "o", color=GRIS, ms=8)
    ax2.plot(lam, mse_1d(x_prueba_1d, y_prueba_1d, lam), "o", color=ROJO, ms=8)
    ax2.set(xlabel=r"$\lambda$", ylabel="MSE", ylim=(0, None))
    ax2.legend(fontsize=9)
    plt.show()

In [ ]:
interact(ver_ridge_1d, lam=widgets.FloatLogSlider(value=0.01, base=10, min=-2, max=3, step=0.05, description="λ"));

## 4. Ridge sobre el polinomio de grado 10 (Figura 4)

Volvemos al polinomio de grado 10, el que peor generalizaba, y lo entrenamos con Ridge. Las columnas están centradas y escaladas con las estadísticas de entrenamiento, igual que en las notas.

Qué observar al mover $\lambda$:

- Con $\lambda$ muy pequeño el polinomio oscila entre los puntos (sobreajuste) y los coeficientes son enormes.
- Cerca del $\lambda$ óptimo de validación (≈ 0.019) el ajuste se parece a la función real, aunque el modelo sigue teniendo 11 parámetros.
- Con $\lambda$ grande el modelo se vuelve demasiado rígido (subajuste).
- La curva de error de la derecha es la de la Figura 1 reflejada: la complejidad disminuye de izquierda a derecha.

In [ ]:
lambdas_rp = np.logspace(-6, 3, 200)
curva_rp = fig03.curva_error_ridge(lambdas_rp)
mse_train_rp = [r["mse_train"] for r in curva_rp]
mse_val_rp = [r["mse_val"] for r in curva_rp]
mejor_lambda_rp = min(curva_rp, key=lambda r: r["mse_val"])["lambda"]


def estilo_lambda(lam):
    if lam < mejor_lambda_rp / 30:
        return dict(color=ROJO, ls="-")
    if lam > mejor_lambda_rp * 30:
        return dict(color=AZUL, ls="--")
    return dict(color=VERDE, ls="-")


def ver_ridge_polinomio(lam=1e-6):
    x = fig03.x_grid
    th = fig03.ridge_cerrada(fig03.X_tr, fig03.y_train, lam)
    fig, (ax1, ax2) = plt.subplots(1, 2)

    ax1.plot(x, fig03.f_real(x), color=GRIS, lw=5, alpha=0.5, label="función real")
    ax1.plot(x, fig03.y_media + fig03.diseno_ridge(x) @ th, lw=2, label="Ridge, grado 10", **estilo_lambda(lam))
    ax1.scatter(fig03.x_train, fig03.y_train, color="black", zorder=3, label="entrenamiento")
    ax1.set(xlim=(0, 5), ylim=(0, 5), xlabel="$x$", ylabel="$y$",
            title=f"λ = {lam:.3g}:  max|θ| = {np.abs(th).max():.3g}")
    ax1.legend(loc="upper left", fontsize=9)

    ax2.loglog(lambdas_rp, mse_train_rp, "--", color=GRIS, lw=2, label="entrenamiento")
    ax2.loglog(lambdas_rp, mse_val_rp, color=ROJO, lw=2, label="validación")
    ax2.axhline(fig03.SIGMA**2, color="0.7", ls=":", label="varianza del ruido")
    ax2.axvline(mejor_lambda_rp, color="0.85", lw=1)
    ax2.plot(lam, fig03.mse_ridge(fig03.X_tr, fig03.y_train, th), "o", color=GRIS, ms=8)
    ax2.plot(lam, fig03.mse_ridge(fig03.X_va, fig03.y_val, th), "o", color=ROJO, ms=8)
    ax2.set(xlabel=r"$\lambda$", ylabel="MSE")
    ax2.legend(fontsize=9)
    plt.show()

In [ ]:
interact(ver_ridge_polinomio, lam=widgets.FloatLogSlider(value=1e-6, base=10, min=-6, max=3, step=0.1, description="λ"));

## 5. Por qué $L_1$ produce ceros y $L_2$ no (Figura 5)

Usamos la formulación con restricción: minimizar $J(\theta)$ sujeto a $\theta_1^2 + \theta_2^2 \le t$ (Ridge, un círculo) o sujeto a $|\theta_1| + |\theta_2| \le t$ (Lasso, un rombo). La × marca el mínimo sin regularizar, $\hat\theta$, y el punto rojo la solución regularizada: el punto de la región donde $J$ es menor. La curva gruesa es la curva de nivel que toca la región.

Qué observar al mover $t$:

- Con Lasso, para un rango grande de valores de $t$ la solución se queda en una esquina del rombo, sobre el eje: $\theta_2 = 0$ exactamente.
- Con Ridge la solución se desliza suavemente por el círculo y nunca cae sobre un eje.
- Si $t$ es suficientemente grande, $\hat\theta$ queda dentro de la región y la restricción deja de importar ($t$ grande equivale a $\lambda$ pequeño).

In [ ]:
theta_ols = fig03.theta_ols
g1, g2 = np.meshgrid(np.linspace(-3, 3.5, 300), np.linspace(-3, 3.5, 300))
dif = np.stack([g1 - theta_ols[0], g2 - theta_ols[1]], axis=-1)
J_malla = np.einsum("...i,ij,...j->...", dif, fig03.A, dif)


def solucion_restringida(region, t):
    if region == "l2":
        dentro, frontera, radio = theta_ols @ theta_ols <= t, fig03.circulo, np.sqrt(t)
    else:
        dentro, frontera, radio = np.abs(theta_ols).sum() <= t, fig03.rombo, t
    if dentro:
        return theta_ols.copy()
    return fig03.minimo_en_frontera(frontera, radio)


def ver_geometria(t=1.0):
    fig, axs = plt.subplots(1, 2, figsize=(11, 5))
    s = np.linspace(0, 2 * np.pi, 400)
    for ax, region, nombre in [(axs[0], "l2", "Ridge ($L_2$)"), (axs[1], "l1", "Lasso ($L_1$)")]:
        radio = np.sqrt(t) if region == "l2" else t
        borde = (fig03.circulo if region == "l2" else fig03.rombo)(s, radio)
        ax.fill(borde[0], borde[1], color=CELESTE, alpha=0.8, ec=AZUL, lw=1.2)
        ax.contour(g1, g2, J_malla, levels=[0.3, 1, 2.5, 5, 9, 15], colors="0.75", linewidths=0.8)
        sol = solucion_restringida(region, t)
        nivel = fig03.J_cuad(sol)
        if nivel > 1e-9:
            ax.contour(g1, g2, J_malla, levels=[nivel], colors="tab:cyan", linewidths=2.5)
        ax.axhline(0, color="0.6", lw=0.8)
        ax.axvline(0, color="0.6", lw=0.8)
        ax.plot(*theta_ols, "x", color="black", ms=10, mew=2.5)
        ax.plot(*sol, "o", color=ROJO, ms=9, zorder=5)
        ax.set(xlim=(-3, 3.5), ylim=(-3, 3.5), aspect="equal", xlabel=r"$\theta_1$", ylabel=r"$\theta_2$",
               title=f"{nombre}:  θ = ({sol[0]:.2f}, {sol[1]:.2f})")
    plt.show()

In [ ]:
interact(ver_geometria, t=widgets.FloatSlider(value=1.0, min=0.05, max=7, step=0.05, description="t"));

## 6. Caminos de coeficientes (Figura 6)

El problema de las casas de las notas: 80 casas con ocho features estandarizadas. Cinco tienen efecto real sobre el precio (área, habitaciones, baños, antigüedad, distancia al centro) y tres son ruido (gris punteado). El área y las habitaciones están correlacionadas.

La línea vertical marca el $\lambda$ actual, y el panel de la derecha muestra los coeficientes de Ridge y Lasso para ese $\lambda$.

Qué observar al mover $\lambda$:

- Con Lasso las tres features de ruido son las primeras en llegar exactamente a cero; el área es la última. Lasso hace selección de features.
- Con Ridge todos los coeficientes se encogen poco a poco, pero ninguno llega a cero.
- El coeficiente de las habitaciones en Ridge crece un poco antes de encogerse: al encoger el área, parte de su peso pasa a la feature correlacionada.

In [ ]:
camino_ridge, camino_lasso = fig03.caminos()
lambdas_c = fig03.lambdas_camino
nombres = fig03.nombres
colores = [AZUL, NARANJA, VERDE, ROJO, "tab:purple"] + [GRIS] * 3


def ver_caminos(lam=1.0):
    fig, axs = plt.subplots(1, 3, figsize=(15, 4.6), gridspec_kw=dict(width_ratios=[1, 1, 1.1]))
    for ax, camino, titulo in [(axs[0], camino_ridge, "Ridge"), (axs[1], camino_lasso, "Lasso")]:
        for j, nombre in enumerate(nombres):
            ax.semilogx(lambdas_c, camino[:, j], color=colores[j], lw=2, ls="--" if j >= 5 else "-",
                        label=nombre if j < 5 else ("Ruido" if j == 5 else None))
        ax.axhline(0, color="0.8", lw=0.8)
        ax.axvline(lam, color="black", lw=1)
        ax.set(xlabel=r"$\lambda$", ylabel="Coeficiente", title=titulo, ylim=(-2.6, 3.4))
    axs[0].legend(fontsize=8, loc="upper right")

    th_r, th_l = fig03.ridge_casas(lam), fig03.lasso_casas(lam)
    pos = np.arange(len(nombres))[::-1]
    axs[2].barh(pos + 0.2, th_r, height=0.38, color=colores, alpha=0.45, label="Ridge")
    axs[2].barh(pos - 0.2, th_l, height=0.38, color=colores, label="Lasso")
    axs[2].axvline(0, color="0.5", lw=0.8)
    axs[2].set_yticks(pos, nombres, fontsize=9)
    axs[2].set(xlim=(-2.6, 3.4), xlabel="Coeficiente",
               title=f"λ = {lam:.3g}:  Lasso usa {np.sum(np.abs(th_l) > 1e-10)} de {len(nombres)} features")
    axs[2].legend(fontsize=9, loc="lower right")
    plt.tight_layout()
    plt.show()

In [ ]:
interact(ver_caminos, lam=widgets.FloatLogSlider(value=1.0, base=10, min=-2, max=4, step=0.05, description="λ"));